# Customer Churn — Data Cleaning, Step by Step

**Dataset:** `Customer_Churn.csv` (telecom churn)
**Target:** `churn` (1 = churned, 0 = retained)

There are no helper functions in this notebook on purpose: every step is plain sequential pandas. Above every code cell there is a short note that says what was done in that step, why it was done, and what the result was.

## Step1: Locate the input file


In [8]:
from pathlib import Path
import numpy as np
import pandas as pd

source = Path("data_raw/Customer_Churn_Cleaned.csv")
churn = pd.read_csv(source)
literal = pd.read_csv(source, dtype=str, keep_default_na=False)
print("Raw shape:", churn.shape)
print(churn.head(3).to_string(index=False))
print(churn.dtypes.to_string())

Raw shape: (2850, 15)
 call_failure  complains  subscription_length  charge_amount  seconds_of_use  frequency_of_use  frequency_of_sms  distinct_called_numbers  tariff_plan  status  age  customer_value  churn  inactive_customer  outlier_flag
            8          0                   38              0            4370                71                 5                       17            1       1   30         197.640      0                  0             0
            0          0                   39              0             318                 5                 7                        4            1       2   25          46.035      0                  0             0
           10          0                   37              0            2453                60               359                       24            1       1   30        1536.520      0                  0             1
call_failure                 int64
complains                    int64
subscription_length         

## Step2: Read the file and take a first look at the data

**What we did:** read `Customer_Churn.csv` with pandas, then printed the number of rows and columns, the column names, and how many customers churned before any cleaning.

**Result:** 3,150 rows x 14 columns, and churn was 495 customers (15.71%).

In [9]:
source = Path("data_raw/Customer_Churn_Cleaned.csv")
df = pd.read_csv(source)
# df = pd.read_csv("C:\\Users\\pppppp\\Desktop\\Customer_Churn_Cleaned.csv")
print("Shape before cleaning:", df.shape, "=", df.shape[0], "rows x", df.shape[1], "columns")
print("Columns before cleaning:", list(df.columns))
print("churn before cleaning:", int(df["churn"].sum()), "churned out of", len(df) ,
      "=", round(float(df["churn"].mean()) * 100, 2), "%")
df.head()

Shape before cleaning: (2850, 15) = 2850 rows x 15 columns
Columns before cleaning: ['call_failure', 'complains', 'subscription_length', 'charge_amount', 'seconds_of_use', 'frequency_of_use', 'frequency_of_sms', 'distinct_called_numbers', 'tariff_plan', 'status', 'age', 'customer_value', 'churn', 'inactive_customer', 'outlier_flag']
churn before cleaning: 446 churned out of 2850 = 15.65 %


,call_failure,complains,subscription_length,charge_amount,seconds_of_use,frequency_of_use,frequency_of_sms,distinct_called_numbers,tariff_plan,status,age,customer_value,churn,inactive_customer,outlier_flag
0,8,0,38,0,4370,71,5,17,1,1,30,197.640,0,0,0
1,0,0,39,0,318,5,7,4,1,2,25,46.035,0,0,0
2,10,0,37,0,2453,60,359,24,1,1,30,1536.520,0,0,1
3,10,0,38,0,4198,66,1,35,1,1,15,240.020,0,0,0
4,3,0,38,0,2393,58,2,33,1,1,15,145.805,0,0,0


## Step 2: Fix the column names

**What I did:** converted all column names to snake_case and removed double spaces, for example `"Call  Failure"` became `"call_failure"`.

**Why:** 3 names had double spaces, which makes the columns awkward to reference in code.

In [10]:
df.columns = [c.strip().lower().replace("  ", " ").replace(" ", "_") for c in df.columns]
print("Columns after renaming:", list(df.columns))

Columns after renaming: ['call_failure', 'complains', 'subscription_length', 'charge_amount', 'seconds_of_use', 'frequency_of_use', 'frequency_of_sms', 'distinct_called_numbers', 'tariff_plan', 'status', 'age', 'customer_value', 'churn', 'inactive_customer', 'outlier_flag']


## Step 3: Check for missing values

**What I did:** checked every cell in the data for missing values, and counted the missing values in each column and in the whole dataset.

**Result:** 0 missing values out of 44,100 cells (3,150 rows x 14 columns).

**Why nothing was filled in:** there are no missing values at all, so filling any would invent data. This step is a check and a report only.

In [11]:
missing_per_column = df.isna().sum()
total_missing = int(missing_per_column.sum())
total_cells = int(df.shape[0] * df.shape[1])
print("Missing values per column:")
print(missing_per_column.to_string())
print("Total missing:", total_missing, "out of", total_cells, "cells")

Missing values per column:
call_failure               0
complains                  0
subscription_length        0
charge_amount              0
seconds_of_use             0
frequency_of_use           0
frequency_of_sms           0
distinct_called_numbers    0
tariff_plan                0
status                     0
age                        0
customer_value             0
churn                      0
inactive_customer          0
outlier_flag               0
Total missing: 0 out of 42750 cells


## Step 4: Remove duplicate rows

**What we did:** removed rows that are identical in every column with `drop_duplicates()`.

**Result:** 300 duplicate rows removed, the data went from 3,150 to 2,850 rows.

**Why:** the same customer counted several times would bias every rate that follows.

In [12]:
rows_before_dedup = len(df)
df = df.drop_duplicates().reset_index(drop=True)
rows_removed = rows_before_dedup - len(df)
print("Duplicate rows removed:", rows_removed)
print("Shape after removing duplicates:", df.shape)
print("Duplicates remaining:", int(df.duplicated().sum()))

Duplicate rows removed: 0
Shape after removing duplicates: (2850, 15)
Duplicates remaining: 0


Step 5: Drop the age_group column

What We Did: Dropped the age_group column and kept the original age column.

Why: The age_group column contained the same information as the age column. Each age group corresponded to exactly one age value, with no exceptions. Therefore, keeping both columns was redundant.

Result: The dataset was reduced from 14 to 13 columns before adding the binary flags

In [13]:
print("Age column still exists:", "age" in churn.columns)
print("Age_group column exists:", "age_group" in churn.columns)

print("Shape after dropping age_group:", churn.shape)

Age column still exists: True
Age_group column exists: False
Shape after dropping age_group: (2850, 15)


## Step 6: Add the inactive_customer flag

**What I did:** added a new column, `inactive_customer` = 1 for customers whose `seconds_of_use` == 0, and 0 for everyone else.

**Result:** 107 customers (3.75% of the cleaned data).

**Why a flag and not a deletion:** zero seconds of use is real behaviour (a customer who subscribed but never used the service), not a data error. All 107 also have `frequency_of_use` == 0 and `distinct_called_numbers` == 0, which confirms the flag is correct.

In [14]:
df["inactive_customer"] = (df["seconds_of_use"] == 0).astype(int)
print("Inactive customers (seconds_of_use == 0):", int(df["inactive_customer"].sum()))

Inactive customers (seconds_of_use == 0): 107


## Step 7: Add the outlier_flag

**What I did:** computed Tukey fences (1.5 x IQR) for each of the 6 continuous columns below, and set `outlier_flag` = 1 for any row outside the fences in any of them.
The 6 columns: `call_failure`, `seconds_of_use`, `frequency_of_use`, `frequency_of_sms`, `distinct_called_numbers`, `customer_value`.

**Result:** 561 rows (19.68%) have at least one outlier value.

**Why a flag and not a deletion or capping:** no value is impossible. Every value is >= 0 and inside a plausible range, so deleting them would throw away real customers. **Why the rule was not applied to the other columns:** on binary / code columns (`complains`, `tariff_plan`, `status`, `churn`) an IQR rule is meaningless and would only flag a code value, and on `subscription_length` (3-47 months) and `charge_amount` (0-10) the values are bounded discrete scores, so a short subscription or a low charge is a valid value, not an error.

In [15]:
continuous_columns = [
    "call_failure",
    "seconds_of_use",
    "frequency_of_use",
    "frequency_of_sms",
    "distinct_called_numbers",
    "customer_value",
]
df["outlier_flag"] = 0
for col in continuous_columns:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    is_outlier = (df[col] < lower) | (df[col] > upper)
    df.loc[is_outlier, "outlier_flag"] = 1
    print(col, ": fence [", round(float(lower), 2), ",", round(float(upper), 2),
          "] outliers =", int(is_outlier.sum()))
print("Rows with outlier_flag = 1:", int(df["outlier_flag"].sum()),
      "=", round(float(df["outlier_flag"].mean()) * 100, 2), "%")

call_failure : fence [ -15.5 , 28.5 ] outliers = 44
seconds_of_use : fence [ -6103.12 , 14061.88 ] outliers = 189
frequency_of_use : fence [ -74.0 , 198.0 ] outliers = 118
frequency_of_sms : fence [ -114.5 , 209.5 ] outliers = 333
distinct_called_numbers : fence [ -23.5 , 68.5 ] outliers = 82
customer_value : fence [ -891.3 , 1798.91 ] outliers = 104
Rows with outlier_flag = 1: 561 = 19.68 %


## Step 8: Final checks and save the cleaned file

**What I did:** checked the final shape of the data, confirmed there are no missing values and no duplicate rows left after cleaning, computed the final churn rate, and saved the cleaned file as `cleaned_dataset.csv`.

**Result:** 2,850 rows x 15 columns, 0 missing, 0 duplicates, and churn is 446 customers (15.65%).

In [16]:
print("Final shape:", df.shape, "=", df.shape[0], "rows x", df.shape[1], "columns")
print("Final columns:", list(df.columns))
print("Missing values after cleaning:", int(df.isna().sum().sum()))
print("Duplicate rows after cleaning:", int(df.duplicated().sum()))
print("churn after cleaning:", int(df["churn"].sum()), "out of", len(df),
      "=", round(float(df["churn"].mean()) * 100, 2), "%")
df.to_csv(source, index=False)
print("Saved:", source)
print("Done.")
df.head()

Final shape: (2850, 15) = 2850 rows x 15 columns
Final columns: ['call_failure', 'complains', 'subscription_length', 'charge_amount', 'seconds_of_use', 'frequency_of_use', 'frequency_of_sms', 'distinct_called_numbers', 'tariff_plan', 'status', 'age', 'customer_value', 'churn', 'inactive_customer', 'outlier_flag']
Missing values after cleaning: 0
Duplicate rows after cleaning: 0
churn after cleaning: 446 out of 2850 = 15.65 %
Saved: data_raw/Customer_Churn_Cleaned.csv
Done.


,call_failure,complains,subscription_length,charge_amount,seconds_of_use,frequency_of_use,frequency_of_sms,distinct_called_numbers,tariff_plan,status,age,customer_value,churn,inactive_customer,outlier_flag
0,8,0,38,0,4370,71,5,17,1,1,30,197.640,0,0,0
1,0,0,39,0,318,5,7,4,1,2,25,46.035,0,0,0
2,10,0,37,0,2453,60,359,24,1,1,30,1536.520,0,0,1
3,10,0,38,0,4198,66,1,35,1,1,15,240.020,0,0,0
4,3,0,38,0,2393,58,2,33,1,1,15,145.805,0,0,0
